# Notebook 34 — Nested probability calibration and coverage-matched entry research

**Question:** Do chronological out-of-fold stacking models select better Up-class entries than a full-history Random Forest **at comparable entry coverage**?

This notebook performs a chronological **inner calibration split** inside each outer training fold. The inner models predict the later calibration period. We fit a binary Platt mapping and choose target-coverage thresholds **using only calibration-period labels/scores**. We then refit the base and meta models on all available outer-training history and apply the frozen calibration mapping and thresholds to the untouched outer validation fold.

**Important:** Calibration mappings are fitted using inner-model predictions, then transferred to separately refitted outer models. Probability distributions may shift; report actual coverage rather than assuming it equals the target. The results are development-only; the previously closed historical holdout and prospective prediction ledger are not accessed.

Entry policy: rank by **class-1 probability alone** (no argmax requirement); subtract 10 bps per selected entry. This policy differs from Notebook 33's `argmax==1` plus fixed probability threshold and should not be compared without accounting for the change.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src').exists():raise FileNotFoundError('Open inside stock_ml_dynamic repository')
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.coverage_calibration_research import run_research
SEED=42
N_OUTER=3
OOF_WINDOWS=5
CAP=60000
TREES=200
LEAF=20
CLIP=(.005,.995)
COVERAGES=(.05,.10,.15,.20,.25)
COST_BPS=10
CALIBRATION_FRACTION=.25
OUTPUT=ROOT/'artifacts'/'notebook34_calibration_coverage'
print('Repository:',ROOT)


## Load the same 1d master-table artifact as Notebook 33

In [ ]:
source=ROOT/'artifacts'/'master_tables'/'1d'
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
missing=[str(source/name) for name in required if not (source/name).exists()]
if missing:raise FileNotFoundError('Missing master-table artifacts: '+', '.join(missing)+'; rerun Notebook 03 from repository root or move notebooks/artifacts/master_tables/1d to artifacts/master_tables/1d')
master,latest,bundle,manifest=load_master(source)
print('Master:',master.shape,'Latest:',latest.shape,'Tickers:',master.symbol.nunique())
print('Date range:',master.bar_timestamp.min(),master.bar_timestamp.max())


## Run nested chronological calibration experiment

Expect roughly twice the model-fitting workload of Notebook 33. The three outer folds each run an inner calibration experiment and an outer validation experiment. No validation labels are used to fit models, calibrators, or thresholds.

In [ ]:
result=run_research(master,n_outer=N_OUTER,windows=OOF_WINDOWS,cap=CAP,trees=TREES,
                    leaf=LEAF,seed=SEED,clip=CLIP,coverages=COVERAGES,
                    cost_bps=COST_BPS,calibration_fraction=CALIBRATION_FRACTION,
                    output_dir=OUTPUT)
policies=result['policies'];audit=result['audit'];reliability=result['reliability'];preds=result['predictions']
print('Development cutoff:',result['cutoff'],'Features:',result['feature_count'])
print('Policy rows:',len(policies),'Validation predictions:',len(preds))
display(audit)


## Chronological leakage assertions

In [ ]:
assert (pd.to_datetime(audit.latest_fit_outcome,utc=True)<pd.to_datetime(audit.earliest_cal_bar,utc=True)).all()
assert (pd.to_datetime(audit.latest_outer_train_outcome,utc=True)<pd.to_datetime(audit.earliest_outer_valid_bar,utc=True)).all()
assert audit.inner_oof_windows.eq(OOF_WINDOWS).all()
assert audit.outer_oof_windows.eq(OOF_WINDOWS).all()
print('PASS: calibration and outer validation are strictly after their respective training labels.')


## Policy comparison at calibration-derived target coverage

In [ ]:
cols=['coverage','up_precision','mean_net','entry_count','hit_rate_net','brier_up','binary_logloss']
summary=policies.groupby(['target_coverage','model','calibration'])[cols].mean().round(6)
display(summary)
for metric in ['up_precision','mean_net','coverage']:
    fig,ax=plt.subplots(figsize=(11,5))
    for (model,calibration),group in policies.groupby(['model','calibration']):
        g=group.groupby('target_coverage')[metric].mean()
        ax.plot(g.index*100,g.values,marker='o',label=f'{model} / {calibration}')
    ax.set(xlabel='Calibration-period target coverage (%)',ylabel=metric,title=f'Validation {metric} at preselected thresholds')
    ax.legend(fontsize=8);ax.grid(alpha=.2);plt.show()


## Paired comparisons with the full-history RF at the same target coverage

These are **target-coverage matched** comparisons, not guaranteed exact realized-coverage matches. Check the actual validation coverage before interpreting precision differences.

In [ ]:
base=policies.loc[policies.model.eq('full_history_rf')].set_index(['fold','target_coverage','calibration'])
paired=[]
for model,g in policies.groupby('model'):
    if model=='full_history_rf':continue
    g=g.set_index(['fold','target_coverage','calibration'])
    for metric in ['up_precision','mean_net','coverage','brier_up','binary_logloss']:
        delta=(g[metric]-base[metric]).dropna()
        for (fold,coverage,cal),value in delta.items():
            paired.append(dict(fold=fold,target_coverage=coverage,calibration=cal,model=model,
                               metric=metric,difference=value))
paired=pd.DataFrame(paired)
display(paired.groupby(['target_coverage','calibration','model','metric']).difference.agg(['mean','min','max']).round(6))


## Calibration reliability curves and Brier scores

In [ ]:
for model in ['full_history_rf','oof_meta_logistic','oof_meta_rf']:
    fig,ax=plt.subplots(figsize=(6,5))
    for variant in ['raw','platt']:
        sub=reliability.loc[(reliability.model==model)&(reliability.subset==variant)]
        agg=sub.groupby('bin').apply(lambda g:pd.Series({
            'n':g.n.sum(),
            'pred':np.average(g.predicted_up.dropna(),weights=g.loc[g.predicted_up.notna(),'n']) if g.predicted_up.notna().any() else np.nan,
            'actual':np.average(g.observed_up.dropna(),weights=g.loc[g.observed_up.notna(),'n']) if g.observed_up.notna().any() else np.nan
        }),include_groups=False)
        ax.plot(agg['pred'],agg['actual'],marker='o',label=variant)
    ax.plot([0,1],[0,1],'k--',alpha=.4)
    ax.set(xlim=(0,1),ylim=(0,1),xlabel='Mean predicted Up probability',ylabel='Observed Up rate',title=model)
    ax.legend();ax.grid(alpha=.2);plt.show()
display(policies.groupby(['model','calibration'])[['brier_up','binary_logloss']].mean().round(6))


## Up precision and net returns by predicted-probability decile

Decile boundaries here are computed on the outer validation scores for **descriptive diagnostics only**. They must not be used as deployed trading thresholds.

In [ ]:
decile_rows=[]
for (fold,model,variant),g in preds.groupby(['fold','model','calibration']):
    g=g.copy()
    g['decile']=pd.qcut(g.up_probability.rank(method='first'),10,labels=False)+1
    for decile,h in g.groupby('decile'):
        decile_rows.append(dict(fold=fold,model=model,calibration=variant,decile=decile,
                                n=len(h),up_precision=(h.direction==1).mean(),
                                mean_net=(h.next_return-COST_BPS/10000).mean(),
                                avg_probability=h.up_probability.mean()))
deciles=pd.DataFrame(decile_rows)
display(deciles.groupby(['model','calibration','decile'])[['up_precision','mean_net','avg_probability']].mean().round(5))
for metric in ['up_precision','mean_net']:
    fig,ax=plt.subplots(figsize=(10,5))
    for model,g in deciles.loc[deciles.calibration.eq('raw')].groupby('model'):
        h=g.groupby('decile')[metric].mean()
        ax.plot(h.index,h.values,marker='o',label=model)
    ax.set(xlabel='Validation probability decile (10 = highest)',ylabel=metric,title=f'Raw model: {metric} by score decile')
    ax.legend();ax.grid(alpha=.2);plt.show()


## Decision checklist

- Compare **paired** Up precision and mean net returns at 5–25% target coverage across all three folds.
- Examine **realized** coverage gaps; large gaps mean threshold transfer is unstable.
- Check whether Platt calibration improves binary Brier/log loss. Calibration alone cannot improve ranking within a fold because sigmoid Platt mapping is monotone.
- Treat validation deciles as exploratory, not as thresholds selected without leakage.
- Do **not** promote a trading strategy based on these development results alone. The historical final holdout stays closed.

In [ ]:
print('Saved results:',OUTPUT)
print('Historical final holdout: not accessed. Prospective prediction ledger: unchanged.')
